In [1]:
%pip install -U trl

Note: you may need to restart the kernel to use updated packages.


In [2]:
from importlib import reload

from Trainers.trainer_ppo import PPOTrainingConfig, PolicyPPOTrainer
from Models.model_policy import PolicyModel
from Models.model_value import ValueModel
from Models.model_reward import RewardModel
import Datasets.dataset_request as dataset_request

dataset_request = reload(dataset_request)
RequestDataset = dataset_request.RequestDataset

c:\Users\misha\anaconda3\envs\torch310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

config = PPOTrainingConfig(
    output_dir="outputs/ppo_policy"
)
policy = PolicyModel("Qwen/Qwen3-0.6B")
value = ValueModel("Qwen/Qwen3-0.6B")
reward_model = RewardModel("Skywork/Skywork-Reward-V2-Qwen3-0.6B", "proxy")
dataset = RequestDataset.load("human_requests_hh-rlhf.pt", "Qwen/Qwen3-0.6B")
dataset.truncate(0, 16)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 7089.03it/s]
Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 4850.93it/s]


In [4]:
policy.generate_new_dataset(dataset, 4)

RequestDataset(size=16)

In [ ]:
answers = policy.generate_batch(data_list)
for question, answer in zip(data_list, answers):
    print(f"Question:\n {question}\n\nAnswer:\n {answer}\n\n")
    

In [ ]:
%pip show trl

In [ ]:
trainer = PolicyPPOTrainer(policy, reward_model, value, dataset, config)

In [ ]:
trainer.train()

In [ ]:
# After training, policy is changed. I want to answers = policy.generate_batch(data_list)
answers = policy.generate_batch(data_list)
for question, answer in zip(data_list, answers):
    print(f"Question:\n {question}\n\nAnswer:\n {answer}\n\n")

In [ ]:
from Models.model_evaluator import PrometheusEvaluator


evaluator = PrometheusEvaluator()
print(evaluator.score(data_list[0], answers[0]))